# EndoScan — MRI Image Classifier
## CRISP-DM Data Science Lifecycle

---
**Project:** EndoScan ESI Score — MRI Imaging Path
**Model:** EfficientNet-B0 Binary Classifier → Probability → ESI Score (1–100) → Tier
**Methodology:** CRISP-DM (6 phases)
**Data:** Real extracted slices (`outputs/mri_slices`) + synthetic augmented slices
(`outputs/augmented/mri`), both keyed by `patient_id`

---
### Architecture Context
```
NLP Path (live)          → free-text symptoms      → ESI score + tier ─┐
Structured Path (live)   → questionnaire answers    → ESI score + tier ─┤
MRI Path (this)          → MRI slice                → ESI score + tier ─┴─► Fusion Layer → Unified ESI
```
This notebook must produce output in the same `esi_score` (1–100) / `esi_tier`
(Low/Moderate/High/Critical) shape as the structured-symptom path so the fusion
layer can combine all three paths. That's new — the earlier version of this
notebook only produced a raw probability.

## Phase 1 — Business Understanding
Define the objective, the primary evaluation metric, and success criteria before touching data.


In [2]:
# Phase 1 — Business Understanding
# No computation here — documented as the project contract

BUSINESS_OBJECTIVE = """
The MRI path ingests a T2-weighted pelvic MRI slice and outputs a
patient-level ESI score and tier, in the same format as the structured
and NLP paths, for the fusion layer.

Primary clinical concern: missing a true endometriosis case (false
negative) is far costlier than a false positive that gets ruled out at
specialist referral. RECALL (sensitivity) is therefore the primary
metric — not AUC, not accuracy. AUC and specificity are tracked as
secondary/diagnostic metrics, not the selection criterion.
"""

ESI_TIERS = {
    'Low':      (1,  25,  'Minimal indicators — routine monitoring'),
    'Moderate': (26, 50,  'Moderate indicators — follow up 4–6 weeks'),
    'High':     (51, 75,  'Strong indicators — specialist referral within 2 weeks'),
    'Critical': (76, 100, 'Severe presentation — urgent specialist referral'),
}

# Recall is PRIMARY. Everything else is diagnostic/secondary.
SUCCESS_CRITERIA = {
    'Recall':      0.80,   # PRIMARY — minimum acceptable sensitivity
    'Specificity': 0.75,   # secondary — floor, not the optimisation target
    'AUC':         0.85,   # secondary — diagnostic only
}

print("Phase 1 — Business Understanding")
print("=" * 50)
print("Primary metric   : Recall (sensitivity)")
print("Secondary floor  : Specificity >= 0.75")
print("Diagnostic only  : AUC")
print(f"\nESI Tier Boundaries:")
for tier, (lo, hi, action) in ESI_TIERS.items():
    print(f"  {tier:<10} {lo:>3}–{hi:<3}  {action}")
print(f"\nSuccess criteria  : Recall ≥ {SUCCESS_CRITERIA['Recall']}  |  "
      f"Specificity ≥ {SUCCESS_CRITERIA['Specificity']}  |  "
      f"AUC ≥ {SUCCESS_CRITERIA['AUC']} (informational)")

Phase 1 — Business Understanding
Primary metric   : Recall (sensitivity)
Secondary floor  : Specificity >= 0.75
Diagnostic only  : AUC

ESI Tier Boundaries:
  Low          1–25   Minimal indicators — routine monitoring
  Moderate    26–50   Moderate indicators — follow up 4–6 weeks
  High        51–75   Strong indicators — specialist referral within 2 weeks
  Critical    76–100  Severe presentation — urgent specialist referral

Success criteria  : Recall ≥ 0.8  |  Specificity ≥ 0.75  |  AUC ≥ 0.85 (informational)


## Phase 2 — Data Understanding
Structural audit of both the real and synthetic slice sets, and EDA before any modelling decision is made.

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────
import json, os, copy, time, random
from pathlib import Path

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import rotate, zoom as scipy_zoom

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    classification_report, confusion_matrix
)
import timm

print('All imports OK')
print(f'PyTorch  : {torch.__version__}')
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Device   : {device_name}')
print(f'timm     : {timm.__version__}')

In [ ]:
# ── Config ───────────────────────────────────────────────────────────
# All paths and hyperparameters in one place.

MRI_ROOT        = Path.home() / 'Downloads/UT-EndoMRI'
PNG_DIR         = Path('./outputs/mri_slices')
RECORDS_PATH    = Path('./outputs/mri_slice_records.json')
CHECKPOINT_PATH = Path('./outputs/mri_best_model.pt')
RESULTS_DIR     = Path('./outputs')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PNG_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 50
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 8
FREEZE_BLOCKS = 6    # freeze blocks 0–5; train block 6 + head (~28% params)
DROPOUT       = 0.3
SEED          = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device         : {DEVICE}')
print(f'MRI_ROOT exists: {MRI_ROOT.exists()}')
print(f'PNG_DIR        : {PNG_DIR}  (exists: {PNG_DIR.exists()})')
print(f'Records cached : {RECORDS_PATH.exists()}')
print(f'Hyperparams    : IMG={IMG_SIZE} | BS={BATCH_SIZE} | LR={LR} | '
      f'EP={EPOCHS} | patience={PATIENCE} | freeze={FREEZE_BLOCKS} blocks')

---
## Phase 2 — Data Understanding: Inspect the NIfTI files

In [ ]:
# ── Scan institutions and print a summary ───────────────────────────
# This cell reads directory metadata only — no large arrays are loaded.

def scan_institution(inst_dir: Path) -> dict:
    """Return {patient_id: {dir, sequences, em_labels, has_em}} for one site."""
    patients = {}
    for pt_dir in sorted(inst_dir.iterdir()):
        if not pt_dir.is_dir():
            continue
        files  = list(pt_dir.glob('*.nii*'))
        seqs   = [f for f in files if any(
                  f.stem.endswith(s) for s in ['_T1','_T2','_T1FS','_T2FS'])]
        labels = [f for f in files if '_em' in f.stem]
        patients[pt_dir.name] = {
            'dir'      : pt_dir,
            'sequences': seqs,
            'em_labels': labels,
            'has_em'   : len(labels) > 0,
        }
    return patients


if MRI_ROOT.exists():
    d1_patients = scan_institution(MRI_ROOT / 'D1_MHS')
    d2_patients = scan_institution(MRI_ROOT / 'D2_TCPW')

    for site, pts in [('D1_MHS', d1_patients), ('D2_TCPW', d2_patients)]:
        pos = sum(1 for p in pts.values() if p['has_em'])
        print(f'{site}: {len(pts)} patients | {pos} have endo masks '
              f'({pos/len(pts)*100:.0f}%)')

    # Show one example patient
    first_id, first_info = next(iter(d1_patients.items()))
    print(f'\nExample patient: {first_id}')
    print(f'  Sequences : {[f.name for f in first_info["sequences"]]}')
    print(f'  EM labels : {[f.name for f in first_info["em_labels"]]}')
else:
    print(f'MRI_ROOT not found at {MRI_ROOT} — skipping institution scan.')
    print('Set MRI_ROOT in the Config cell and rerun.')
    d1_patients, d2_patients = {}, {}

In [ ]:
# ── Visualise sample axial slices with and without endo mask ────────
# Loads ONE patient volume to show what the raw data looks like.

def show_sample_slices(pt_info: dict, pt_id: str, n_slices: int = 5):
    """Plot n_slices evenly-spaced axial T2 slices, overlaying mask when present."""
    t2_files = [f for f in pt_info['sequences'] if f.stem.endswith('_T2')]
    if not t2_files:
        print(f'No T2 for {pt_id}'); return

    vol  = nib.load(t2_files[0]).get_fdata()
    n_sl = vol.shape[2]
    idxs = np.linspace(0, n_sl - 1, n_slices, dtype=int)

    # Try to load a mask
    mask = None
    if pt_info['em_labels']:
        m = nib.load(pt_info['em_labels'][0]).get_fdata()
        if m.shape != vol.shape:
            factors = [t/s for t, s in zip(vol.shape, m.shape)]
            m = scipy_zoom(m, factors, order=0)
        mask = (m > 0).astype(np.float32)

    fig, axes = plt.subplots(1, n_slices, figsize=(3 * n_slices, 3.5))
    for ax, idx in zip(axes, idxs):
        sl = vol[:, :, idx]
        ax.imshow(sl.T, cmap='gray', origin='lower')
        if mask is not None:
            m_sl = mask[:, :, idx]
            if m_sl.max() > 0:
                ax.contour(m_sl.T, levels=[0.5], colors='red', linewidths=1.5)
        label = 'ENDO' if (mask is not None and mask[:,:,idx].max()>0) else 'NEG'
        ax.set_title(f'sl {idx}\n{label}', fontsize=8)
        ax.axis('off')
    fig.suptitle(f'Patient {pt_id} — T2 axial (red = endo contour)', fontsize=10)
    plt.tight_layout()
    plt.show()
    print(f'Volume shape: {vol.shape}  |  '
          f'Voxel size: {nib.load(t2_files[0]).header.get_zooms()}')


if d1_patients:
    # Pick a patient that has endo for a more interesting visualisation
    endo_pts = {k: v for k, v in d1_patients.items() if v['has_em']}
    sample_id = next(iter(endo_pts)) if endo_pts else next(iter(d1_patients))
    show_sample_slices(d1_patients[sample_id], sample_id)
else:
    print('Skipping visualisation — MRI_ROOT not available.')

---
## Phase 3 — Data Preparation

In [ ]:
# ── Consensus mask builder ───────────────────────────────────────────
# D1 (3 raters): positive if ≥ 2 raters agree.
# D2 (1 rater) : positive if mask > 0.

def resample_mask(mask: np.ndarray, target_shape: tuple) -> np.ndarray:
    factors = [t / s for t, s in zip(target_shape, mask.shape)]
    return scipy_zoom(mask, factors, order=0)   # nearest-neighbour


def get_consensus_mask(pt_info: dict, target_shape: tuple, threshold: int = 2):
    """
    Returns float32 binary mask aligned to target_shape, or None.
    threshold : minimum rater agreement to call a voxel positive.
    """
    em_files = sorted([f for f in pt_info['em_labels'] if '_em_r' in f.stem])
    if not em_files:
        em_files = pt_info['em_labels']
    if not em_files:
        return None

    if len(em_files) == 1:
        m = nib.load(em_files[0]).get_fdata()
        if m.shape != target_shape:
            m = resample_mask(m, target_shape)
        return (m > 0).astype(np.float32)

    # Multi-rater: vote map
    vote_map = np.zeros(target_shape, dtype=np.float32)
    for f in em_files:
        m = nib.load(f).get_fdata()
        if m.shape != target_shape:
            m = resample_mask(m, target_shape)
        vote_map += (m > 0).astype(np.float32)
    return (vote_map >= threshold).astype(np.float32)


print('Consensus mask functions defined OK')

In [ ]:
# ── Slice-level record builder ───────────────────────────────────────
# Produces a flat list of dicts; one per axial T2 slice.
# PNG files are written here so the Dataset cell only needs imread().

def extract_slices_and_save_pngs(patients: dict, institution: str,
                                  png_dir: Path, is_d1: bool = True) -> list:
    """
    For each patient, load the T2 volume, compute consensus mask,
    label every axial slice, and save it as a PNG.
    Returns a list of slice-level record dicts.
    """
    records = []
    for pt_id, info in patients.items():
        t2_files = [f for f in info['sequences'] if f.stem.endswith('_T2')]
        if not t2_files:
            print(f'  [skip] {pt_id} — no T2')
            continue

        t2_vol   = nib.load(t2_files[0]).get_fdata()
        t2_shape = t2_vol.shape
        mask     = get_consensus_mask(info, t2_shape,
                                      threshold=2 if is_d1 else 1)

        # Normalise full volume to [0, 255] for PNG
        v_min, v_max = t2_vol.min(), t2_vol.max()
        vol_norm = ((t2_vol - v_min) / (v_max - v_min + 1e-8) * 255).astype(np.uint8)

        for sl_idx in range(t2_shape[2]):
            label = int(mask is not None and mask[:, :, sl_idx].max() > 0)
            fname = f'{pt_id}_sl{sl_idx:03d}_label{label}.png'
            png_path = png_dir / fname
            if not png_path.exists():          # don't overwrite on re-run
                sl = vol_norm[:, :, sl_idx]
                sl_resized = np.array(
                    Image.fromarray(sl).resize((IMG_SIZE, IMG_SIZE),
                                               Image.BILINEAR))
                Image.fromarray(sl_resized).save(png_path)

            records.append({
                'patient_id' : pt_id,
                'institution': institution,
                'slice_idx'  : sl_idx,
                'label'      : label,
                't2_path'    : str(t2_files[0]),
                'is_d1'      : is_d1,
            })

        print(f'  {pt_id}: {t2_shape[2]} slices | '
              f'{sum(1 for r in records[-t2_shape[2]:] if r["label"]==1)} pos')

    return records


print('Slice extractor defined OK')

In [ ]:
# ── Build or load slice records ──────────────────────────────────────
# If the JSON cache exists, skip the slow NIfTI scan entirely.

if RECORDS_PATH.exists():
    with open(RECORDS_PATH) as f:
        all_records = json.load(f)
    print(f'Loaded {len(all_records):,} records from cache ({RECORDS_PATH})')
else:
    if not MRI_ROOT.exists():
        raise FileNotFoundError(
            f'MRI_ROOT not found at {MRI_ROOT}.\n'
            'Download UT-EndoMRI and update MRI_ROOT in the Config cell.')

    print('Building records from NIfTI — this may take a few minutes...')
    print('\nD1_MHS:')
    d1_records = extract_slices_and_save_pngs(
        d1_patients, 'D1_MHS', PNG_DIR, is_d1=True)
    print('\nD2_TCPW:')
    d2_records = extract_slices_and_save_pngs(
        d2_patients, 'D2_TCPW', PNG_DIR, is_d1=False)

    all_records = d1_records + d2_records
    with open(RECORDS_PATH, 'w') as f:
        json.dump(all_records, f, indent=2)
    print(f'\nSaved {len(all_records):,} records -> {RECORDS_PATH}')

# ── Summary stats ────────────────────────────────────────────────────
n_total = len(all_records)
n_pos   = sum(r['label'] == 1 for r in all_records)
n_neg   = n_total - n_pos
n_pts   = len(set(r['patient_id'] for r in all_records))
print(f'\nDataset summary')
print(f'  Total slices : {n_total:,}')
print(f'  Positive     : {n_pos:,} ({n_pos/n_total*100:.1f}%)')
print(f'  Negative     : {n_neg:,} ({n_neg/n_total*100:.1f}%)')
print(f'  Imbalance    : 1:{n_neg//max(n_pos,1)}')
print(f'  Patients     : {n_pts}')

In [ ]:
# ── Patient-level split ──────────────────────────────────────────────
# Splits by patient ID — NEVER by slice — to prevent data leakage.

def patient_split(records: list, train_frac=0.70, val_frac=0.15, seed=42):
    all_ids = list(set(r['patient_id'] for r in records))
    rng = random.Random(seed)
    rng.shuffle(all_ids)

    n_train   = int(len(all_ids) * train_frac)
    n_val     = int(len(all_ids) * val_frac)
    train_ids = set(all_ids[:n_train])
    val_ids   = set(all_ids[n_train:n_train + n_val])
    test_ids  = set(all_ids[n_train + n_val:])

    # Strict leakage check
    assert train_ids.isdisjoint(val_ids),  'LEAK: train/val overlap!'
    assert train_ids.isdisjoint(test_ids), 'LEAK: train/test overlap!'
    assert val_ids.isdisjoint(test_ids),   'LEAK: val/test overlap!'

    train = [r for r in records if r['patient_id'] in train_ids]
    val   = [r for r in records if r['patient_id'] in val_ids]
    test  = [r for r in records if r['patient_id'] in test_ids]
    return train, val, test


train_records, val_records, test_records = patient_split(
    all_records, seed=SEED)

print('Patient-level split (no leakage):')
for name, recs in [('Train', train_records), ('Val', val_records),
                   ('Test',  test_records)]:
    n   = len(recs)
    pos = sum(r['label'] == 1 for r in recs)
    pts = len(set(r['patient_id'] for r in recs))
    print(f'  {name:5s}: {pts:3d} patients | {n:6,} slices | '
          f'{pos:5,} pos ({pos/n*100:.1f}%)')

In [ ]:
# ── Visualise label distribution ─────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors = ['steelblue', 'coral']

for ax, (name, recs) in zip(axes,
    [('Train', train_records), ('Val', val_records), ('Test', test_records)]):
    n     = len(recs)
    n_pos = sum(r['label'] == 1 for r in recs)
    n_neg = n - n_pos
    ax.bar(['Negative', 'Positive'], [n_neg, n_pos], color=colors)
    ax.set_title(f'{name}  (n={n:,})')
    ax.set_ylabel('Slices')
    for i, v in enumerate([n_neg, n_pos]):
        ax.text(i, v + n*0.01, f'{v:,}\n({v/n*100:.0f}%)',
                ha='center', fontsize=9)

plt.suptitle('Class distribution per split', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Class distribution plot saved.')

---
## Phase 3 — Data Preparation: Dataset & DataLoaders

In [ ]:
# ── Augmentation (geometry-only) ─────────────────────────────────────
# MRI intensities are clinically meaningful — colour/brightness jitter
# would corrupt the signal, so only spatial transforms are applied.

def augment_slice(sl: np.ndarray) -> np.ndarray:
    if np.random.rand() > 0.5:
        sl = np.fliplr(sl)
    if np.random.rand() > 0.3:
        sl = np.flipud(sl)
    if np.random.rand() > 0.5:
        sl = rotate(sl, np.random.uniform(-15, 15), reshape=False, order=1)
    if np.random.rand() > 0.5:
        f  = np.random.uniform(0.9, 1.1)
        z  = scipy_zoom(sl, f, order=1)
        h, w   = sl.shape
        zh, zw = z.shape
        if f > 1.0:
            sl = z[(zh-h)//2:(zh-h)//2+h, (zw-w)//2:(zw-w)//2+w]
        else:
            ph, pw = (h-zh)//2, (w-zw)//2
            sl = np.pad(z, ((ph, h-zh-ph), (pw, w-zw-pw)), mode='constant')
    return sl.astype(np.float32)


# ── Dataset ──────────────────────────────────────────────────────────
class MRISliceDataset(Dataset):
    """
    Loads pre-saved PNG slices.  PNG filename convention:
        {patient_id}_sl{slice_idx:03d}_label{label}.png
    """
    def __init__(self, records, png_dir, augment=False, img_size=IMG_SIZE):
        self.records  = records
        self.png_dir  = Path(png_dir)
        self.augment  = augment
        self.img_size = img_size

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec   = self.records[idx]
        fname = (f"{rec['patient_id']}_"
                 f"sl{rec['slice_idx']:03d}_"
                 f"label{rec['label']}.png")
        sl = np.array(Image.open(self.png_dir / fname).convert('L'),
                      dtype=np.float32)
        sl = sl / 255.0 * 2.0 - 1.0          # [0,255] → [-1, 1]

        if self.augment:
            sl = augment_slice(sl)

        # Resize only if PNG dimensions differ from IMG_SIZE
        if sl.shape[0] != self.img_size or sl.shape[1] != self.img_size:
            sl = scipy_zoom(sl,
                            (self.img_size / sl.shape[0],
                             self.img_size / sl.shape[1]), order=1)

        sl_3ch = np.stack([sl, sl, sl], axis=0)    # greyscale → 3-channel
        return (torch.tensor(sl_3ch, dtype=torch.float32),
                torch.tensor(rec['label'], dtype=torch.float32))


print('Dataset defined OK')

# Quick smoke-test
if PNG_DIR.exists() and any(PNG_DIR.glob('*.png')):
    ds_test = MRISliceDataset(train_records[:4], PNG_DIR, augment=False)
    img, lbl = ds_test[0]
    print(f'Sample tensor: shape={tuple(img.shape)} | '
          f'min={img.min():.2f} max={img.max():.2f} | label={int(lbl)}')
else:
    print('No PNGs yet — run the extraction cell above first.')

In [ ]:
# ── DataLoaders with weighted sampling ───────────────────────────────
# WeightedRandomSampler ensures every training batch is ~50/50 pos/neg
# regardless of the raw 1:11 imbalance.

train_ds = MRISliceDataset(train_records, PNG_DIR, augment=True)
val_ds   = MRISliceDataset(val_records,   PNG_DIR, augment=False)
test_ds  = MRISliceDataset(test_records,  PNG_DIR, augment=False)

# Per-sample weights inversely proportional to class frequency
train_labels = [r['label'] for r in train_records]
n_pos_train  = sum(train_labels)
n_neg_train  = len(train_labels) - n_pos_train
pos_weight   = n_neg_train / n_pos_train      # used later in BCEWithLogitsLoss

sample_weights = torch.tensor(
    [1.0/n_pos_train if l == 1 else 1.0/n_neg_train for l in train_labels],
    dtype=torch.float32)
sampler = WeightedRandomSampler(sample_weights, len(sample_weights),
                                replacement=True)

NUM_WORKERS = 0   # set to 2–4 on Linux/Colab for speed

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)

print(f'pos_weight for loss = {pos_weight:.2f}  '
      f'(n_pos={n_pos_train}, n_neg={n_neg_train})')
print(f'Train : {len(train_loader):3d} batches | {len(train_ds):,} samples')
print(f'Val   : {len(val_loader):3d} batches | {len(val_ds):,} samples')
print(f'Test  : {len(test_loader):3d} batches | {len(test_ds):,} samples')

In [ ]:
# ── Visualise augmented training samples ─────────────────────────────

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
pos_shown = neg_shown = 0

for rec in train_records:
    if pos_shown >= 6 and neg_shown >= 6:
        break
    row = 0 if rec['label'] == 1 else 1
    col = pos_shown if rec['label'] == 1 else neg_shown
    if (rec['label'] == 1 and pos_shown >= 6) or \
       (rec['label'] == 0 and neg_shown >= 6):
        continue

    fname = (f"{rec['patient_id']}_sl{rec['slice_idx']:03d}"
             f"_label{rec['label']}.png")
    png_p = PNG_DIR / fname
    if not png_p.exists():
        continue

    sl = np.array(Image.open(png_p).convert('L'), dtype=np.float32) / 255.0
    axes[row, col].imshow(sl, cmap='gray', vmin=0, vmax=1)
    axes[row, col].axis('off')
    if rec['label'] == 1:
        pos_shown += 1
    else:
        neg_shown += 1

axes[0, 0].set_ylabel('Endo (pos)', rotation=90, labelpad=8, fontsize=10)
axes[1, 0].set_ylabel('No endo (neg)', rotation=90, labelpad=8, fontsize=10)
plt.suptitle('Sample training slices', fontsize=12)
plt.tight_layout()
plt.show()

---
## Phase 4 — Modelling

In [ ]:
# ── Build EfficientNet-B0 ────────────────────────────────────────────
# Blocks 0–5 frozen (ImageNet features), block 6 + head trainable (~28%).
# Single-logit output → BCEWithLogitsLoss (numerically stable).

model = timm.create_model('efficientnet_b0', pretrained=True)

# Freeze all
for param in model.parameters():
    param.requires_grad = False

# Unfreeze block 6 onwards
for block in model.blocks[FREEZE_BLOCKS:]:
    for param in block.parameters():
        param.requires_grad = True

# Unfreeze conv_head and BN
for m in [model.conv_head, model.bn2]:
    for param in m.parameters():
        param.requires_grad = True

# Replace classifier
in_feat = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT),
    nn.Linear(in_feat, 1)
)

model = model.to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total:,}')
print(f'Trainable parameters : {trainable:,}  ({trainable/total*100:.1f}%)')
print(f'Frozen parameters    : {total-trainable:,}  ({(total-trainable)/total*100:.1f}%)')

In [ ]:
# ── Training infrastructure ──────────────────────────────────────────
# Loss: BCEWithLogitsLoss with pos_weight to penalise missed endo.
# Optimiser: Adam with weight decay.
# Scheduler: CosineAnnealingLR.
# Early stopping: halts on val loss plateau and restores best weights.

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight]).to(DEVICE))

optimizer = Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# ── Early stopping state ─────────────────────────────────────────────
best_val_loss  = float('inf')
patience_count = 0
best_weights   = copy.deepcopy(model.state_dict())
MIN_DELTA      = 1e-4

# ── Training history ─────────────────────────────────────────────────
# FIX: all keys populated inside the loop — the original code missed
# train_loss, train_acc, val_loss, val_acc, val_auc.
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss'  : [], 'val_acc'  : [], 'val_auc': [],
    'val_recall': [], 'val_precision': [],
    'val_f1'    : [], 'val_specificity': [],
}

print(f'Loss      : BCEWithLogitsLoss  pos_weight={pos_weight:.2f}')
print(f'Optimiser : Adam  lr={LR}  wd={WEIGHT_DECAY}')
print(f'Scheduler : CosineAnnealingLR  T_max={EPOCHS}')
print(f'Patience  : {PATIENCE} epochs')
print('Training infrastructure ready.')

In [ ]:
# ── Training loop ────────────────────────────────────────────────────
# Prints a rich metrics row every epoch.
# Saves checkpoint whenever val loss improves.
# Restores best weights at the end.

print(f'Training | max {EPOCHS} epochs | patience={PATIENCE}')
print('-' * 82)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Train one epoch ──────────────────────────────────────────────
    model.train()
    tr_loss = tr_correct = tr_total = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        tr_loss    += loss.item() * imgs.size(0)
        tr_correct += ((torch.sigmoid(out) > 0.5).float() == labels).sum().item()
        tr_total   += imgs.size(0)

    train_loss = tr_loss / tr_total
    train_acc  = tr_correct / tr_total * 100

    # ── Validate ─────────────────────────────────────────────────────
    model.eval()
    vl_loss = vl_correct = vl_total = 0
    val_probs_ep, val_labels_ep = [], []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE).unsqueeze(1)
            out   = model(imgs)
            loss  = criterion(out, labels)
            probs = torch.sigmoid(out).cpu().squeeze(1)
            vl_loss    += loss.item() * imgs.size(0)
            vl_correct += ((probs > 0.5).float() ==
                           labels.cpu().squeeze(1)).sum().item()
            vl_total   += imgs.size(0)
            val_probs_ep.extend(probs.numpy())
            val_labels_ep.extend(labels.cpu().squeeze(1).numpy())

    val_loss = vl_loss / vl_total
    val_acc  = vl_correct / vl_total * 100

    vp = np.array(val_probs_ep)
    vl = np.array(val_labels_ep)
    vd = (vp > 0.5).astype(float)

    try:    val_auc = roc_auc_score(vl, vp)
    except: val_auc = float('nan')

    tp = ((vd == 1) & (vl == 1)).sum()
    fp = ((vd == 1) & (vl == 0)).sum()
    fn = ((vd == 0) & (vl == 1)).sum()
    tn = ((vd == 0) & (vl == 0)).sum()

    precision   = tp / (tp + fp)  if (tp + fp) > 0  else float('nan')
    recall      = tp / (tp + fn)  if (tp + fn) > 0  else float('nan')
    f1          = (2 * precision * recall / (precision + recall)
                   if not np.isnan(precision) and not np.isnan(recall)
                   and (precision + recall) > 0 else float('nan'))
    specificity = tn / (tn + fp)  if (tn + fp) > 0  else float('nan')

    # ── Append ALL history keys (the original bug fixed here) ────────
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    history['val_recall'].append(recall)
    history['val_precision'].append(precision)
    history['val_f1'].append(f1)
    history['val_specificity'].append(specificity)

    scheduler.step()

    # ── Checkpoint ───────────────────────────────────────────────────
    improved = val_loss < best_val_loss - MIN_DELTA
    if improved:
        best_val_loss = val_loss
        best_weights  = copy.deepcopy(model.state_dict())
        patience_count = 0
        torch.save({'epoch': epoch,
                    'model_state_dict': best_weights,
                    'val_loss': val_loss,
                    'val_auc' : val_auc,
                    'history' : history},
                   CHECKPOINT_PATH)
    else:
        patience_count += 1

    tag = '  ← best' if improved else f'  (patience {patience_count}/{PATIENCE})'
    print(f'Ep {epoch:3d}/{EPOCHS} | '
          f'tr_loss {train_loss:.4f}  tr_acc {train_acc:.1f}% | '
          f'val_loss {val_loss:.4f}  val_acc {val_acc:.1f}% | '
          f'auc {val_auc:.3f}  rec {recall:.3f}  f1 {f1:.3f}  '
          f'spec {specificity:.3f} | '
          f'{time.time()-t0:.0f}s{tag}')

    if patience_count >= PATIENCE:
        print(f'\nEarly stopping triggered at epoch {epoch}.')
        break

# ── Restore best weights ─────────────────────────────────────────────
model.load_state_dict(best_weights)
best_ep = history['val_loss'].index(min(history['val_loss'])) + 1
print(f'\nTraining complete.')
print(f'  Best epoch : {best_ep}')
print(f'  Best val loss : {min(history["val_loss"]):.4f}')
print(f'  Best val AUC  : {max(history["val_auc"]):.4f}')

In [ ]:
# ── Training curves ──────────────────────────────────────────────────

epochs_ran = len(history['val_loss'])
x = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

axes[0].plot(x, history['train_loss'], label='Train')
axes[0].plot(x, history['val_loss'],   label='Val')
axes[0].axvline(best_ep, color='red', linestyle='--', linewidth=1,
                label=f'Best ep {best_ep}')
axes[0].set_title('BCE Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('Epoch')

axes[1].plot(x, history['train_acc'], label='Train')
axes[1].plot(x, history['val_acc'],   label='Val')
axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('Epoch')

axes[2].plot(x, history['val_auc'],         label='AUC')
axes[2].plot(x, history['val_recall'],      label='Recall')
axes[2].plot(x, history['val_specificity'], label='Specificity')
axes[2].plot(x, history['val_f1'],          label='F1')
axes[2].set_title('Val Metrics'); axes[2].legend(); axes[2].grid(alpha=0.3)
axes[2].set_xlabel('Epoch')

axes[3].plot(x, history['val_precision'], label='Precision')
axes[3].plot(x, history['val_recall'],    label='Recall')
axes[3].set_title('Precision vs Recall'); axes[3].legend(); axes[3].grid(alpha=0.3)
axes[3].set_xlabel('Epoch')

plt.suptitle('EfficientNet-B0 — Training History', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

---
## Phase 5 — Evaluation

In [ ]:
# ── Test set inference ───────────────────────────────────────────────

model.eval()
all_probs, all_labels_test = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        probs = torch.sigmoid(model(imgs.to(DEVICE))).cpu().squeeze(1)
        all_probs.extend(probs.numpy())
        all_labels_test.extend(labels.numpy())

all_probs       = np.array(all_probs)
all_labels_test = np.array(all_labels_test)
all_preds       = (all_probs > 0.5).astype(float)

# ── Metrics ──────────────────────────────────────────────────────────
try:    test_auc = roc_auc_score(all_labels_test, all_probs)
except: test_auc = float('nan')

cm = confusion_matrix(all_labels_test, all_preds)
if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan')
else:
    sensitivity = specificity = float('nan')

accuracy = (all_preds == all_labels_test).mean() * 100

print('=' * 60)
print('  TEST SET EVALUATION — MRI (EfficientNet-B0)')
print('=' * 60)
n_test = len(all_labels_test)
n_pos_test = int(all_labels_test.sum())
print(f'  Samples      : {n_test} ({n_pos_test} pos / {n_test-n_pos_test} neg)')
print(f'  AUC-ROC      : {test_auc:.4f}   (target ≥ 0.85)')
print(f'  Accuracy     : {accuracy:.1f}%')
print(f'  Sensitivity  : {sensitivity:.3f}   (recall for endo, target ≥ 0.80)')
print(f'  Specificity  : {specificity:.3f}   (recall for no-endo, target ≥ 0.75)')
print()
try:
    print(classification_report(
        all_labels_test, all_preds,
        target_names=['No endo', 'Endo'], digits=3))
except ValueError as e:
    print(f'classification_report error: {e}')
print('=' * 60)

# ── Clinical success check ───────────────────────────────────────────
print('\nClinical targets:')
for metric, val, threshold in [
    ('AUC-ROC',     test_auc,    0.85),
    ('Sensitivity', sensitivity, 0.80),
    ('Specificity', specificity, 0.75),
]:
    status = '✅ MET' if val >= threshold else '❌ NOT MET'
    print(f'  {metric:12s}: {val:.3f}  (≥{threshold})  {status}')

In [ ]:
# ── Evaluation plots ─────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# 1. ROC curve
try:
    fpr, tpr, thresholds = roc_curve(all_labels_test, all_probs)
    axes[0].plot(fpr, tpr, lw=2, color='steelblue',
                 label=f'AUC = {test_auc:.3f}')
    axes[0].plot([0, 1], [0, 1], '--', color='gray', linewidth=1)
    axes[0].axhline(sensitivity, color='coral', linestyle=':',
                    label=f'Sensitivity = {sensitivity:.3f}')
    axes[0].set_xlabel('1 - Specificity (FPR)')
    axes[0].set_ylabel('Sensitivity (TPR)')
    axes[0].set_title('ROC Curve')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
except Exception as e:
    axes[0].set_title(f'ROC error: {e}')

# 2. Confusion matrix
im = axes[1].imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=axes[1])
classes = ['No Endo', 'Endo']
tick_marks = [0, 1]
axes[1].set_xticks(tick_marks); axes[1].set_xticklabels(classes)
axes[1].set_yticks(tick_marks); axes[1].set_yticklabels(classes)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[1].text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]*100:.0f}%)',
                     ha='center', va='center',
                     color='white' if cm_norm[i,j] > 0.5 else 'black')
axes[1].set_title('Confusion Matrix')
axes[1].set_ylabel('True label')
axes[1].set_xlabel('Predicted label')

# 3. Confidence histogram
pos_p = all_probs[all_labels_test == 1]
neg_p = all_probs[all_labels_test == 0]
axes[2].hist(neg_p, bins=20, alpha=0.65, label='No endo', color='steelblue')
axes[2].hist(pos_p, bins=20, alpha=0.65, label='Endo',    color='coral')
axes[2].axvline(0.5, color='red', linestyle='--', linewidth=1.5,
                label='Threshold 0.5')
axes[2].set_title('Predicted Probability')
axes[2].set_xlabel('P(endo)')
axes[2].legend()
axes[2].grid(alpha=0.3)

# 4. Sensitivity vs Specificity across thresholds
try:
    sensitivities = tpr
    specificities = 1 - fpr
    axes[3].plot(thresholds, sensitivities[:-1], label='Sensitivity', color='coral')
    axes[3].plot(thresholds, specificities[:-1], label='Specificity', color='steelblue')
    axes[3].axvline(0.5, color='gray', linestyle='--', linewidth=1)
    axes[3].set_title('Sens / Spec vs Threshold')
    axes[3].set_xlabel('Decision threshold')
    axes[3].legend()
    axes[3].grid(alpha=0.3)
except Exception:
    axes[3].set_title('Threshold plot N/A')

plt.suptitle('EfficientNet-B0 — Test Set Results (MRI)', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'mri_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Evaluation plots saved -> outputs/mri_results.png')

In [ ]:
# ── Grad-CAM visualisation ───────────────────────────────────────────
# Highlights which image regions drove each prediction.
# Hooks into the last convolutional block.

activations, gradients = {}, {}

def forward_hook(module, inp, out):
    activations['feat'] = out.detach()

def backward_hook(module, grad_in, grad_out):
    gradients['feat'] = grad_out[0].detach()

# Hook onto the final conv block
target_layer = model.blocks[-1][-1]
fwd_handle   = target_layer.register_forward_hook(forward_hook)
bwd_handle   = target_layer.register_full_backward_hook(backward_hook)


def grad_cam(img_tensor: torch.Tensor) -> np.ndarray:
    """Return a [0,1] heatmap the same size as the input slice."""
    model.eval()
    img = img_tensor.unsqueeze(0).to(DEVICE).requires_grad_(True)
    out = model(img)
    model.zero_grad()
    out.backward()

    grads = gradients['feat'].squeeze(0)         # C, H, W
    acts  = activations['feat'].squeeze(0)       # C, H, W
    weights = grads.mean(dim=(1, 2), keepdim=True)
    cam   = (weights * acts).sum(dim=0).cpu().numpy()
    cam   = np.maximum(cam, 0)                   # ReLU
    if cam.max() > 0:
        cam /= cam.max()
    cam = np.array(Image.fromarray((cam * 255).astype(np.uint8))
                   .resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0
    return cam


# ── Show Grad-CAM on 4 test slices (2 pos, 2 neg) ────────────────────
pos_recs = [r for r in test_records if r['label'] == 1][:2]
neg_recs = [r for r in test_records if r['label'] == 0][:2]
sample_recs = pos_recs + neg_recs

fig, axes = plt.subplots(2, len(sample_recs), figsize=(4 * len(sample_recs), 5))

for col, rec in enumerate(sample_recs):
    fname = (f"{rec['patient_id']}_sl{rec['slice_idx']:03d}"
             f"_label{rec['label']}.png")
    png_p = PNG_DIR / fname
    if not png_p.exists():
        continue

    raw  = np.array(Image.open(png_p).convert('L'), dtype=np.float32)
    sl   = raw / 255.0 * 2.0 - 1.0
    tens = torch.tensor(np.stack([sl, sl, sl], 0), dtype=torch.float32)
    cam  = grad_cam(tens)

    prob = torch.sigmoid(model(tens.unsqueeze(0).to(DEVICE))).item()
    title = f'True: {"Endo" if rec["label"]==1 else "No endo"}\n'\
            f'P(endo)={prob:.2f}'

    axes[0, col].imshow(raw / 255.0, cmap='gray')
    axes[0, col].set_title(title, fontsize=8)
    axes[0, col].axis('off')

    axes[1, col].imshow(raw / 255.0, cmap='gray')
    axes[1, col].imshow(cam, cmap='jet', alpha=0.45)
    axes[1, col].set_title('Grad-CAM', fontsize=8)
    axes[1, col].axis('off')

plt.suptitle('Grad-CAM — Top: original | Bottom: attention map', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'gradcam.png', dpi=150, bbox_inches='tight')
plt.show()

fwd_handle.remove()
bwd_handle.remove()
print('Grad-CAM saved -> outputs/gradcam.png')

---
## Phase 6 — Save final model

In [ ]:
# ── Save checkpoint with full metadata ───────────────────────────────

torch.save({
    'model_state_dict': model.state_dict(),
    'history'         : history,
    'train_patients'  : list(set(r['patient_id'] for r in train_records)),
    'val_patients'    : list(set(r['patient_id'] for r in val_records)),
    'test_patients'   : list(set(r['patient_id'] for r in test_records)),
    'metrics': {
        'auc'        : float(test_auc),
        'accuracy'   : float(accuracy),
        'sensitivity': float(sensitivity),
        'specificity': float(specificity),
    },
    'config': {
        'img_size'     : IMG_SIZE,
        'batch_size'   : BATCH_SIZE,
        'epochs_run'   : len(history['val_loss']),
        'lr'           : LR,
        'freeze_blocks': FREEZE_BLOCKS,
        'dropout'      : DROPOUT,
        'modality'     : 'MRI',
        'model'        : 'efficientnet_b0',
    }
}, CHECKPOINT_PATH)

print(f'Checkpoint saved  -> {CHECKPOINT_PATH}')
print(f'AUC-ROC           : {test_auc:.4f}')
print(f'Sensitivity       : {sensitivity:.3f}')
print(f'Specificity       : {specificity:.3f}')
print(f'Accuracy          : {accuracy:.1f}%')
print(f'Epochs trained    : {len(history["val_loss"])}')
print(f'Best epoch        : {best_ep}')
print()
print('Outputs written:')
for f in sorted(RESULTS_DIR.glob('*')):
    print(f'  {f}')

In [1]:
import json
from collections import Counter

with open('outputs/mri_slice_records.json') as f:
    records = json.load(f)

print("Total records:", len(records))
print("Institutions:", Counter(r['institution'] for r in records))
print("is_d1 values:", Counter(r['is_d1'] for r in records))

# Cross-tab: does is_d1 always match institution, or do they diverge?
cross = Counter((r['institution'], r['is_d1']) for r in records)
for k, v in cross.items():
    print(k, "→", v)

Total records: 4242
Institutions: Counter({'D2_TCPW': 2299, 'D1_MHS': 1943})
is_d1 values: Counter({False: 2299, True: 1943})
('D1_MHS', True) → 1943
('D2_TCPW', False) → 2299
